# Семинар 5 - ARM

Теперь поговорим про ARM ассемблер (конкретнее, про AArch64). Процессоры на базе архитектуры ARM устанавливаются в компьютеры Apple, а также большинство портативной электроники (смартфонов, планшетов и так далее). Архитектура ощутимо эффективнее x86, поэтому многие считают, что за ним будущее.

Есть 2 подхода к проектированию архитектур компьютера: CISC и RISC.
* CISC - Complex Instruction Set Computer - подход, при котором в архитектуре поддержан богатый набор инструкций. Например, x86
* RISC - Reduced Instruction Set Computer - подход, при котором предоставляется только базовый набор инструкций. 

Исторически был более распространен первый подход. Например, в x86 есть инструкции `PCLMULQDQ` (Perform a carry-less multiplication of two 64-bit polynomials over the finite field $GF_{2^k}$) и `PCMPxSTRx` (perform arithmetic comparisons between all possible pairs of bytes or words, one from each packed input source operand). Из-за того, что инструкций много, но при этом некоторые из них являются очень узкоспециализированными, в x86 используется переменная длинна инструкций: от 1 до 15 байт. [Здесь](https://www.felixcloutier.com/x86/) можно посмотретьдлину различных инструкций.

Минус такого подхода состоит в том, что это мешает пайплайнингу процессора. 

<img src="media/pipeline.png" alt="cpu pipelining" width="400" style="background-color:white;"/>

Процессор на самом деле работает с несколькими инструкциями одновременно, когда это возможно. В RISC подходе, в силу того, что количество инструкций сильно ограничено, можно добиться того, что все инструкции будут иметь одинаковую длину (arm). Благодаря этому, первые 2 этапа (fetch и decode) пайплайнинга можно реализовать намного более эффективно.

В реальности CISC архитектуры довольно часто используют [микро-инструкции](https://en.wikipedia.org/wiki/Micro-operation), имеющие одинаковую длину, для нивелирования этой разницы.

## Настройка окружения


Если у вас Linux, Необходимо скачать компилятор и библиотеки для другой платформы из проекта [Linaro](http://releases.linaro.org/components/toolchain/binaries/7.5-2019.12/aarch64-linux-gnu/). Полезно в `~/.bashrc` дописать алиасы наподобие:

```
alias agcc='/opt/aarch64-gcc/bin/aarch64-linux-gnu-gcc'
alias agdb='/opt/aarch64-gcc/bin/aarch64-linux-gnu-gdb'
alias aobjdump='/opt/aarch64-gcc/bin/aarch64-linux-gnu-objdump'
alias arun='qemu-aarch64 -L /opt/aarch64-sysroot'
```

Если Mac на Apple Silicon чипе, все должно работать и так

## Регистры и соглашения о вызовах

В `armv8` есть 31 64-битный регистр общего назначения, доступных программно: `x0`, `x1`,  ... , `x30`. Через `w0`, `w1`, ..., `w30` можно обращаться к младшим 32 битам этих регистров. 

В системе Linux предусмотрены следующие соглашения об использовании регистров:

| Регистры        | Назначение                                                   |
| --------------- | ------------------------------------------------------------ |
| `x0` ... `x7`   | аргументы функции и возвращаемое значение (`x0`)             |
| `x8` ... `x18`  | временные регистры, для которые не гарантируется сохранение результата, если вызывать какую-либо функцию |
| `x19` ... `x28` | регистры, для которых гарантируется, что вызываемая функция их не будет портить |
| `x29`           | указатель на границу фрейма функции, обычно используется отладчиком |
| `x30`      | адрес возврата из функции                                    |
| `sp`      | указатель на вершину стека                                   |

Команда `mov a b` копирует в регистр `a` данные из регистра `b`. Для работы с памятью, в отличии от `x86_64`, предусмотрены специальные инструкции `ldr/str register [address]`. Первая загружает данные в регистр, а вторая в память. Как и в случае `mov` в `x86_64`, для оперирования числами меньшего размера есть специальные суффиксы: `ldrb` загружает `uint8_t`, `ldrsb` загружает `int8_t` и так далее (суффикс `h` используется для 2 байтов, `w` для 4 байтов). Пример использования можно посмотреть в `arr_get.S`.

## Арифметические команды


Ещё в нашем минимальном примере можно увидеть некоторые арифметические команды. У большинства из них вид `command res, left, right`. Этот синтаксис означает `res = left + right` где `+` -- произвольная операция: `add`, `sub`, `mul`, `and`, `orr` и так далее. Довольно полезная инструкция `madd res, mleft, mright, anum` делает `mres=mleft*mright+anum`.


## Метки и переходы


Аналогично `x86_64` можно создавать метки и переходить по ним. Для безусловного перехода предназначена команда `b label`. Команда `cmp` сравнивает числа и выставляет специальные флаги. На основе этих флагов работают суффиксы, которые можно дописать к команде `b`:

* `eq`, `ne` (проверка на равенство)
* `lt`, `le`, `gt`, `ge` (сравнение знаковых чисел)
* `hi`,`ls` (> и <= для беззнаковых )

Пример использования можно посмотреть в `sum_n.S`.

## Вызов функций


Для вызова функций есть команда `bl label`. Она кладёт в `x30` адрес следующей команды. При этом нужно не забыть сохранить исходный адрес возврата (например на стек). Для выхода из функции предназначена команда `ret`. Обратите внимание, что встроенных команд `push/pop` в `aarch64` нет, но можно реализовать их самим с помощью макросов. Пример можно посмотреть в `int_echo.S`.

* [ARM Instruction Set](https://iitd-plos.github.io/col718/ref/arm-instructionset.pdf)